In [ ]:
import itertools as it
import os
import subprocess

import numpy as np
import tqdm
from ase import Atoms
from ase.io import write
from monty.serialization import dumpfn
from rdkit import Chem
from rdkit.Chem import AllChem


### nano reactor sandboxing (work in progress) 

In [ ]:
from arcskin.nanoreactor_run import SMILESPack 

compounds = {
    "NO2":  "[N+](=O)[O-]",
    "NO":   "[N]=O",
    "O2":   "O=O",
}

combination = ["NO","O2"]
sp = SMILESPack(["[N]=O","O=O"])
cells, charges = sp.run_multiple(5,counts=[2,2],min_dist=2) # two of each molecule, with a minimum distance of 2\AA
cells

[Atoms(symbols='NONO5', pbc=False),
 Atoms(symbols='NONO5', pbc=False),
 Atoms(symbols='NONO5', pbc=False),
 Atoms(symbols='NONO5', pbc=False),
 Atoms(symbols='NONO5', pbc=False)]

In [ ]:
class XTBNanoReactor:

    def __init__(self):
        self.input_file = """
    $md
       temp=1200
       time=10.0
       step=0.5
       dump=5.0
       nvt=true
       shake=0
       restart=false
    $end
    $scc
       temp=3000.0
    $end
    $wall
       potential=logfermi
       beta=6.0
       temp=6000.0
       sphere: 14.0, all
    $end
    """




In [ ]:
compounds = {
    "NO2":  "[N+](=O)[O-]",
    "NO":   "[N]=O",
    "O2":   "O=O",
}

combination = ["NO","O2"]

### 1. run a nanoreactor with xtb for a list of compound combinations 

In [ ]:
compounds = {
    "NO2":  "[N+](=O)[O-]",
    "NO":   "[N]=O",
    #"N2":   "N#N",
    "O2":   "O=O",
    #"HNO3": "O[N+](=O)[O-]",
    #"HNO2": "ON=O",
    #"H2":   "[H][H]",
    #"H2O":  "O",
    #"H2O2": "OO",
    #"SO2":"O=S=O",
    #"H2S":"S",
    #"H2SO4":"OS(=O)(=O)O"
}
combinations = list(it.combinations(compounds,2))
combinations

In [ ]:
input_file ="""
    $md
       temp=1200
       time=10.0
       step=0.5
       dump=5.0
       nvt=true
       shake=0
       restart=false
    $end
    $scc
       temp=3000.0
    $end
    $wall
       potential=logfermi
       beta=6.0
       temp=6000.0
       sphere: 14.0, all
    $end
    """

In [ ]:
def smiles_to_atoms(smi, seed=42):
    """Embed a SMILES string in 3D with ETKDG + MMFF and return (Atoms, charge)."""
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smi!r}")
    mol = Chem.AddHs(mol)
    charge = Chem.GetFormalCharge(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    if AllChem.EmbedMolecule(mol, params) != 0:
        raise RuntimeError(f"Embedding failed for {smi!r}")
    AllChem.MMFFOptimizeMolecule(mol)
    atoms = Atoms(
        symbols=[a.GetSymbol() for a in mol.GetAtoms()],
        positions=mol.GetConformer().GetPositions(),
    )
    atoms.positions -= atoms.get_center_of_mass()
    return atoms, charge

def _random_rotation(rng):
    """Uniformly distributed rotation matrix via a random unit quaternion."""
    q = rng.normal(size=4)
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ])

def _point_in_ball(rng, radius):
    v = rng.normal(size=3)
    return v / np.linalg.norm(v) * radius * rng.random() ** (1 / 3)

def pack_smiles(smiles, counts=None, radius=None, min_dist=2.0, contact_dist=4.0,
                max_tries=300, seed=None, filename="structure.xyz", write_chrg=True):
    """Randomly pack molecules into a connected, non-periodic cluster.

    smiles       : list of SMILES strings
    counts       : copies of each (default: one each)
    radius       : cluster radius in Angstrom (default: estimated from molecule sizes)
    min_dist     : minimum interatomic distance between molecules
    contact_dist : each new molecule must come within this of the existing cluster so
                   nothing is left floating in vacuum (set None to disable)

    Returns (Atoms, total_charge).
    """
    rng = np.random.default_rng(seed)
    counts = [1] * len(smiles) if counts is None else list(counts)

    templates = []
    for smi in smiles:
        mol, chg = smiles_to_atoms(smi, seed=int(rng.integers(1 << 30)))
        templates.append(
            (mol, chg, np.linalg.norm(mol.positions, axis=1).max()))

    if radius is None:  # sphere sized for a loose ~35% packing of bounding spheres
        vol = sum(n * (r + min_dist / 2) ** 3 for (_, _, r),
                  n in zip(templates, counts))
        radius = (vol / 0.35) ** (1 / 3)

    system = Atoms()
    charge = 0

    for (template, chg, mol_r), n in zip(templates, counts):
        for copy in range(n):
            for attempt in range(max_tries):
                pos = template.positions @ _random_rotation(rng).T
                pos = pos + _point_in_ball(rng, max(radius - mol_r, 0.0))

                if len(system) == 0:
                    break
                dmin = np.linalg.norm(
                    system.positions[None, :, :] - pos[:, None, :], axis=-1
                ).min()
                if dmin >= min_dist and (contact_dist is None or dmin <= contact_dist):
                    break
                if attempt and attempt % 100 == 0:
                    radius *= 1.1  # too tight, let the cluster breathe
            else:
                raise RuntimeError(
                    f"Could not place copy {copy + 1} of a molecule after {max_tries} "
                    f"tries - lower min_dist or raise radius/contact_dist"
                )

            mol = template.copy()
            mol.positions = pos
            system += mol
            charge += chg

    system.positions -= system.get_center_of_mass()
    write(filename, system, format="xyz",
          comment=f"random cluster, {len(system)} atoms, charge {charge}")
    if write_chrg and charge != 0:
            with open(".CHRG", "w") as f:
                f.write(f"{charge}\n")
    return system, charge

In [ ]:
for r_index,combination in tqdm.tqdm(enumerate(combinations),total=len(combinations),desc="combination",position=0, bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}',leave=True):
    _pwd = os.getcwd()
    directory = os.path.join(_pwd,f"combination_{r_index}")
    os.makedirs(directory)
    # make a json with the combination for reference
    dumpfn(
            {r_index:combination},
        os.path.join(directory,"combination.json"), indent=4
    )
    # change directory to the combination you want to run
    os.chdir(os.path.join(_pwd, f"combination_{r_index}"))
    # get the smiles strings 
    smiles_list = [compounds[p] for p in combination]
    # write the input file
    with open(os.path.join(directory,"md.inp"),"w") as f:
            f.write(input_file)
    
    charges = [
                pack_smiles(
            smiles_list,
            counts=[1, 1],
            min_dist=2.2,
            seed=i,
            filename=f"{directory}/structure_{i}.xyz"
        )[1] for i in range(0, 7)
                ]
    # set up 4 MD runs  
    procs = []
    for i in range(4):
        log = open(f"./out_{i}.log", "w")
        p = subprocess.Popen(
                ["xtb", f"structure_{i}.xyz", "--md", "--input", "md.inp",
                 "--gfn", "2", "--chrg", str(charges[0]), "--uhf", "0", "-P", "4", "--namespace", str(i)],
                stdout=log, stderr=subprocess.STDOUT,
            )
        procs.append((p, log))
    
    for p, log in procs:
        p.wait()
        log.close()
    # go back  
    os.chdir(_pwd)

now using the analysis 

In [ ]:
from ase import Atoms 
from ase.io import read 
from openbabel import openbabel as ob 
from monty.serialization import loadfn 
from collections import Counter,defaultdict
from tqdm.auto import tqdm 
import pandas as pd 

from ase import Atoms
from ase.io import read
from openbabel import openbabel as ob
from monty.serialization import loadfn
from collections import Counter, defaultdict
from python_utils import raise_exception
from tqdm.auto import tqdm
from pymatgen.io.ase import AseAtomsAdaptor
from pymatgen.symmetry.analyzer import PointGroupAnalyzer, iterative_symmetrize
import warnings
from tblite.ase import TBLite
from ase.optimize import BFGS

# uses the same as XTB


class AnalyseNanoReactor:
    def __init__(self, reactants: list = None,skip_step:int = 10):
        warnings.simplefilter("ignore")
        self.trajectory = defaultdict(dict)
        self.unique_structures = defaultdict(lambda: defaultdict(list))
        self.unique_smiles_counter = defaultdict(lambda: defaultdict(list))
        self.unique_composition_counter = defaultdict(lambda: defaultdict(list))
        self.unique_pointgroup_counter = defaultdict(lambda: defaultdict(list))

        self.skip_step = skip_step
        if reactants:
            self.reactants = reactants

    def read_trajectory(self, trajectory: list | str):
        "reads one trajectory of multiple trajectories in xyz format (or xtb's .trj format )"
        if isinstance(trajectory, list):
            for i, traj in enumerate(trajectory):
                self.trajectory[i] = read(traj, format="xyz", index=f"::{self.skip_step}")
        elif isinstance(trajectory, str):
            self.trajectory[0] = read(trajectory, format="xyz", index=f"::{self.skip_step}")

    def relax_atoms(self, atoms: Atoms, fmax=0.01, steps=200):
        mult = 2 if sum(atoms.numbers) % 2 else 1
        atoms.calc = TBLite(method="GFN2-xTB", multiplicity=mult, verbosity=0)
        BFGS(atoms, logfile=None).run(fmax=0.1, steps=200)
        return atoms

    def seperate_atoms(self, atoms: Atoms) -> list[Atoms]:
        """Split an ase.Atoms object into its disconnected molecular fragments."""
        mol = ob.OBMol()
        for Z, pos in zip(atoms.numbers, atoms.positions):
            a = mol.NewAtom()
            a.SetAtomicNum(int(Z))
            a.SetVector(*map(float, pos))
        mol.ConnectTheDots()  # guess bonds from interatomic distances

        return [
            Atoms(numbers=[a.GetAtomicNum() for a in ob.OBMolAtomIter(frag)],
                  positions=[(a.GetX(), a.GetY(), a.GetZ()) for a in ob.OBMolAtomIter(frag)])
            for frag in mol.Separate()
        ]

    def to_obmol(self, atoms: Atoms) -> ob.OBMol:
        mol = ob.OBMol()
        for Z, pos in zip(atoms.numbers, atoms.positions):
            a = mol.NewAtom()
            a.SetAtomicNum(int(Z))
            a.SetVector(*map(float, pos))
        mol.ConnectTheDots()
        mol.PerceiveBondOrders()  # needed for correct double/aromatic bonds
        return mol

    def fragment_smiles(self, atoms: Atoms, fmt: str = "can") -> list[str]:
        """Canonical SMILES for each disconnected fragment."""
        conv = ob.OBConversion()
        conv.SetOutFormat(fmt)
        return [conv.WriteString(frag).split()[0] for frag in self.to_obmol(atoms).Separate()]

    def symmetrize(self, atoms: Atoms, tolerance: float = 0.3) -> tuple[Atoms, str]:
        """Return a symmetrised copy of a molecule and its point group."""
        mol = AseAtomsAdaptor.get_molecule(atoms, charge_spin_check=False)
        sym_mol = iterative_symmetrize(
            mol, max_n=10, tolerance=tolerance)["sym_mol"]
        pg = PointGroupAnalyzer(sym_mol, tolerance=tolerance).sch_symbol
        return AseAtomsAdaptor.get_atoms(sym_mol), pg

    def analyse_trajectory(self, index: int = 0, tolerance: float = 0.3, tqdm_kws={"disable": False}):
        for i, frame in tqdm(enumerate(self.trajectory[index]), total=len(self.trajectory[index]), **tqdm_kws):
            atoms_list = self.seperate_atoms(frame)
            for at in atoms_list:
                try:
                    at = self.relax_atoms(at)
                    sym, pg = self.symmetrize(at, tolerance=tolerance)
                    frags = self.fragment_smiles(sym)
                except Exception:
                    pass
                if len(frags) == 1:
                    self.unique_structures[index]["snapshot_atoms"].append(at)
                    self.unique_structures[index]["symmetrised_atoms"].append(
                        sym)
                    self.unique_structures[index]["compositions"].append(
                        sym.get_chemical_formula())
                    self.unique_structures[index]["smiles"].append(frags[0])
                    self.unique_structures[index]["point_group"].append(pg)

        self.unique_smiles_counter[index] = Counter(
            self.unique_structures[index]["smiles"])
        self.unique_composition_counter[index] = Counter(
            self.unique_structures[index]["compositions"])
        self.unique_pointgroup_counter[index] = Counter(
            self.unique_structures[index]["point_group"])

        if self.reactants:
            indexes_a = [i for i, v in enumerate(
                self.unique_structures[0]["compositions"]) if v == self.reactants[0]]
            indexes_b = [i for i, v in enumerate(
                self.unique_structures[0]["compositions"]) if v == self.reactants[1]]
            indexes_to_delete = indexes_a + indexes_b

            df = pd.DataFrame(self.unique_structures[0])

            df = df.drop(index=indexes_to_delete)
            self.unique_structures[index]["snapshot_atoms"] = df["snapshot_atoms"].values
            self.unique_structures[index]["symmetrised_atoms"] = df["symmetrised_atoms"].values
            self.unique_structures[index]["compositions"] = df["compositions"].values
            self.unique_structures[index]["smiles"] = df["smiles"].values
            self.unique_structures[index]["point_group"] = df["point_group"].values
            self.unique_smiles_counter[index] = Counter(
                self.unique_structures[index]["smiles"])
            self.unique_composition_counter[index] = Counter(
                self.unique_structures[index]["compositions"])
            self.unique_pointgroup_counter[index] = Counter(
                self.unique_structures[index]["point_group"])

    def analyse_trajectories(self, tolerance: float = 0.3, tqdm_kws={"disable": False}):
        if len(self.trajectory) > 1:
            for ii in self.trajectory:
                self.analyse_trajectory(
                    index=ii, tolerance=tolerance, tqdm_kws=tqdm_kws)
        else:
            print("Only one trajectory present, correct usage is AnalyseNanoReactor().analyse_trajectory. Running now...")
            self.analyse_trajectory(tolerance=tolerance, tqdm_kws=tqdm_kws)

In [ ]:
anr = AnalyseNanoReactor(reactants = loadfn("combination_0/combination.json")[str(0)])

In [ ]:
anr.read_trajectory(["combination_0/0.xtb.trj","combination_0/1.xtb.trj"])

In [ ]:
anr.analyse_trajectories()

In [ ]:
dict(anr.unique_smiles_counter)